# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：**24012451**  
第5天专题（A/B/C/D/E）：**A**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [14]:
import sys
!{sys.executable} -m pip install matplotlib


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import matplotlib.pyplot as plt
import matplotlib
print("matplotlib 版本：", matplotlib.__version__)

matplotlib 版本： 3.11.0


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "24012451"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


学生： 24012451
专题： A
输出： output\day06_visualization


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [17]:
# 定义分组函数
def map_tenure_group(tenure):
    if tenure == 0:
        return "新用户"
    elif 1 <= tenure <= 6:
        return "0-6个月"
    elif 7 <= tenure <= 12:
        return "7-12个月"
    elif 13 <= tenure <= 24:
        return "13-24个月"
    else:
        return "24个月以上"

# 对原始数据进行分组（假设你原始的注册时长字段叫Tenure）
df["TenureGroup"] = df["Tenure"].apply(map_tenure_group)

# 题目要求的TENURE_ORDER，原样保留
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
ordered_field = "TenureGroup"

# 按分组汇总
ordered_summary = (
    df.groupby(ordered_field, observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失率=("Churn", "mean")
    )
    .reindex(TENURE_ORDER)
    .reset_index()
)

# 断言校验（和题目完全一致）
assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)

NameError: name 'df' is not defined

In [ ]:
# TODO：填写4个业务问题和图表选择理由
business_questions = {
    "category_bar": "不同用户任期分组（TenureGroup）的用户流失率是否存在差异？",
    "behavior_scatter": "用户任期时长与月度消费金额呈现什么关系？",
    "ordered_line": "用户流失率如何随用户生命周期任期阶段变化？",
    "composition_chart": "全部活跃用户由哪些用户任期（TenureGroup）类别构成？"
}

# 填写对应图表选择理由
chart_reasons = {
    "category_bar": "用户任期分组是离散类别，目标是对比各组流失率指标差异，因此选择柱状图",
    "behavior_scatter": "要分析任期时长和消费金额两个连续变量的相关关系，因此选择散点图",
    "ordered_line": "任期阶段是有序递进的时间阶段，要观察流失率随阶段的连续变化趋势，因此选择折线图",
    "composition_chart": "要展示全部用户在不同任期类别上的占比构成情况，因此选择饼图/环形构成图"
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")


检查点1B通过：业务问题和选择理由已填写


## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
# 题目要求的TENURE_ORDER，原样保留
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
ordered_field = "TenureGroup"

# 直接按数据里的分组汇总，然后按题目顺序重排
ordered_summary = (
    df.groupby(ordered_field, observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失率=("Churn", "mean")
    )
    # 这里只做reindex，不做任何额外赋值
    .reindex(TENURE_ORDER)
    .reset_index()
)

# 断言校验（完全和题目一致）
assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)

display(ordered_summary)

,TenureGroup,用户数,流失率
0,新用户,508,0.54
1,0-6个月,1642,0.26
2,7-12个月,1584,0.10
3,13-24个月,1467,0.06
4,24个月以上,429,0.00


In [ ]:
# TODO: 绘制并保存柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

# 1. 这里直接用你前面的 ordered_summary 表
bars = ax_bar.bar(
    x=ordered_summary["TenureGroup"],
    height=ordered_summary["流失率"],
    color="#1f77b4"
)

# 2. 添加标签（流失率+样本量）
for bar, rate, count in zip(bars, ordered_summary["流失率"], ordered_summary["用户数"]):
    ax_bar.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.01,
        f"{rate:.1%}\n(n={int(count)})",
        ha="center", va="bottom"
    )

# 3. 美化图表
ax_bar.set_title("不同用户任期阶段的流失率对比", fontsize=14)
ax_bar.set_xlabel("用户任期分组（TenureGroup）", fontsize=12)
ax_bar.set_ylabel("用户流失率", fontsize=12)
ax_bar.yaxis.set_major_formatter(PercentFormatter(1))
ax_bar.grid(axis="y", alpha=0.3)

bar_path = OUTPUT_DIR / "01_category_bar.png"
# 4. 保存图片
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.close(fig_bar)

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(OUTPUT_DIR))

已输出： 01_category_bar.png


### 柱状图结论

- 观察：用户流失率随着用户任期时长的增加，呈现出持续下降的趋势。其中，新用户（0个月）流失率最高，随着使用时长的增加，流失风险快速降低，24个月以上的老用户流失率为0%，几乎不存在流失风险。
- 证据：新用户组流失率53.5%（样本量n=508）
0-6个月组流失率25.9%（样本量n=1642）
7-12个月组流失率9.8%（样本量n=1584）
13-24个月组流失率6.5%（样本量n=1467）
24个月以上组流失率0.0%（样本量n=429）
新用户组的流失率比0-6个月组高出27.6个百分点，比7-12个月组高出43.7个百分点，差距非常显著。
- 边界：该图仅能呈现用户任期分组与流失率的相关性，无法证明任期长短是流失率变化的唯一因果因素。用户消费能力、使用频次、套餐类型等其他变量同样会影响流失率，因此不能直接预判单个用户未来是否会流失。

## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
# TODO: 选择两个数值字段，这里用和你专题匹配的任期时长 vs 返现金额
x_field = "Tenure"
y_field = "CashbackAmount"

# 断言校验（保持原任务要求）
assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

# 1. 创建画布
fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

# 2. 计算两类用户样本量，用于图例标注
churn_counts = df["Churn"].value_counts()
churn0_n = churn_counts.get(0, 0)
churn1_n = churn_counts.get(1, 0)

# 3. 按Churn分组绘制散点图，设置透明度避免遮挡
# 已流失用户（红色）
df_churn1 = df[df["Churn"] == 1]
ax_scatter.scatter(
    df_churn1[x_field], df_churn1[y_field],
    color="#d62728", alpha=0.5, label=f"已流失用户 (n={churn1_n})"
)

# 留存用户（蓝色）
df_churn0 = df[df["Churn"] == 0]
ax_scatter.scatter(
    df_churn0[x_field], df_churn0[y_field],
    color="#1f77b4", alpha=0.5, label=f"留存用户 (n={churn0_n})"
)

# ------------------- x轴间隔改为2的关键代码 -------------------
max_x = int(df[x_field].max())
# 步长设为2，生成0,2,4,...的刻度
ax_scatter.set_xticks(range(0, max_x + 2, 2))
# 给x轴两端留边距，避免点被边缘挡住
ax_scatter.set_xlim(left=-1, right=max_x + 1)
# -------------------------------------------------------------

# 4. 美化图表：标题、坐标轴、图例、网格
ax_scatter.set_title("用户任期时长与返现金额的关系（按流失状态区分）", fontsize=14)
ax_scatter.set_xlabel("用户任期时长（月）", fontsize=12)
ax_scatter.set_ylabel("累计返现金额", fontsize=12)
ax_scatter.legend()
ax_scatter.grid(alpha=0.3)

# 5. 保存图片（必须执行，否则断言会报错）
scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.close(fig_scatter)  # 关闭画布，避免Jupyter内存占用

# 断言校验（保持原任务要求）
assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(OUTPUT_DIR))

已输出： 02_behavior_scatter.png


### 散点图结论

- 观察：用户任期时长与累计返现金额整体呈弱正相关关系，随着用户任期增加，返现金额的分布整体上移；已流失用户主要集中在低任期（0-10个月）区间，留存用户则覆盖了更长的任期范围，且返现金额的分布更广泛。
- 证据：1.相关性：随着用户任期从0个月增加到30个月，返现金额的中位数从约150元提升至约220元，呈现明显的上移趋势。
2.聚集特征：已流失用户（红色点）几乎全部集中在0-20个月区间，样本量n=948；留存用户（蓝色点）在 10-30个月区间高度聚集，同时在50-60个月的超长任期也有分布，样本量n=4682。
3.异常点：部分低任期（0-10个月）用户的返现金额超过300元，与多数同任期用户形成差异；超长任期（50-60个月）用户的返现金额也呈现两极分化，存在明显的离群点。
- 边界：该图仅能展示用户任期时长与返现金额的相关关系，无法证明二者存在因果关系，也不能排除用户消费频次、套餐类型等其他变量对返现金额和用户流失状态的影响。

## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
# 题目要求的TENURE_ORDER，必须显式设置
TENURE_ORDER = [
    "新用户", "0-6个月", "7-12个月",
    "13-24个月", "24个月以上"
]

# TODO: 准备有序绘图数据
ordered_field = "TenureGroup"

# 第一步：按真实数据分组汇总
temp_summary = df.groupby(ordered_field, observed=True).agg(
    用户数=("CustomerID", "nunique"),
    流失率=("Churn", "mean")
).reset_index()

# 第二步：手动映射到题目要求的分组（解决名称不匹配问题）
# 封装取值函数，分组不存在则返回0，防止索引报错
def get_val(df, group, col):
    res = df[df[ordered_field] == group]
    return res[col].iloc[0] if not res.empty else 0

ordered_summary = pd.DataFrame({
    ordered_field: TENURE_ORDER,
    "用户数": [
        get_val(temp_summary, "新用户", "用户数"),
        get_val(temp_summary, "0-6个月", "用户数"),
        get_val(temp_summary, "7-12个月", "用户数"),
        get_val(temp_summary, "13-24个月", "用户数"),
        get_val(temp_summary, "24个月以上", "用户数")
    ],
    "流失率": [
        get_val(temp_summary, "新用户", "流失率"),
        get_val(temp_summary, "0-6个月", "流失率"),
        get_val(temp_summary, "7-12个月", "流失率"),
        get_val(temp_summary, "13-24个月", "流失率"),
        get_val(temp_summary, "24个月以上", "流失率")
    ]
})

# 断言校验（和题目完全一致）
assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)



,TenureGroup,用户数,流失率
0,新用户,508,0.54
1,0-6个月,1642,0.26
2,7-12个月,1584,0.10
3,13-24个月,1467,0.06
4,24个月以上,429,0.00


In [ ]:
# TODO: 绘制折线图；若绘制流失率，应标注比例和样本量
fig_line, ax_line = plt.subplots(figsize=(10, 6))

# 绘制折线图，带数据点标记
ax_line.plot(
    ordered_summary[ordered_field],
    ordered_summary["流失率"],
    marker='o',
    linewidth=2,
    color="#ff7f0e"
)

# 给每个点添加标注：流失率百分比 + 样本量
for idx, (tenure, rate, count) in enumerate(
    zip(ordered_summary[ordered_field], ordered_summary["流失率"], ordered_summary["用户数"])
):
    ax_line.text(
        idx, rate + 0.015,
        f"{rate:.1%}\n(n={int(count)})",
        ha="center", va="bottom", fontsize=10
    )

# 美化图表（标题必须说明是有序阶段比较）
ax_line.set_title("用户流失率随任期阶段变化的有序比较", fontsize=14)
ax_line.set_xlabel("用户任期阶段（有序）", fontsize=12)
ax_line.set_ylabel("用户流失率", fontsize=12)
ax_line.yaxis.set_major_formatter(PercentFormatter(1))
ax_line.grid(alpha=0.3)

# 保存路径：适配 notebooks/output 平级结构
from pathlib import Path
# 这里用 ../output/ 回到上级目录再进入 output
line_path = Path("../output/day06_visualization/03_ordered_line.png")
# 确保目录存在
line_path.parent.mkdir(parents=True, exist_ok=True)

fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.close(fig_line)

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.resolve())

已输出： C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day06_visualization\03_ordered_line.png


### 折线图结论

- 观察：用户流失率会随着任职期增长呈现持续下降的规律，新用户阶段流失风险最高，用户留存时间越长，流失概率越低，任职满24个月及以上的用户几乎不会流失。
- 证据：各阶段流失率与样本量依次为：新用户流失率53.5%（n=508）、0-6个月流失率25.9%（n=1642）、7-12个月流失率9.8%（n=1584）、13-24个月流失率6.5%（n=1467）、24个月以上流失率0.0%（n=429）。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势。


## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
# TODO: 选择构成字段并准备汇总表
composition_field = "TenureGroup"
composition_summary = None

# 1. 生成汇总表：按任期分组，统计用户数、计算流失率
composition_summary = df.groupby(composition_field, observed=True).agg(
    用户数=("CustomerID", "nunique"),
    流失率=("Churn", "mean")
).reset_index()

# 2. 计算占比（题目要求保留该列）
total_users = composition_summary["用户数"].sum()
composition_summary["占比"] = composition_summary["用户数"] / total_users

# 题目要求的断言校验
assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"

display(composition_summary)


,TenureGroup,用户数,流失率,占比
0,0-6个月,1642,0.26,0.29
1,13-24个月,1467,0.06,0.26
2,24个月以上,429,0.00,0.08
3,7-12个月,1584,0.10,0.28
4,新用户,508,0.54,0.09


In [ ]:
# TODO: 类别不超过5个时绘制环形图，否则绘制柱状图
from pathlib import Path
fig_composition, ax_composition = plt.subplots(figsize=(10, 10))

# 绘制环形图
wedges, texts, autotexts = ax_composition.pie(
    composition_summary["用户数"],
    labels=composition_summary[composition_field],
    autopct='',
    startangle=90,
    wedgeprops=dict(width=0.4, edgecolor='white'),
    colors=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
)

# 标注：人数 + 占比 + 流失率，字体黑色
for i, txt in enumerate(autotexts):
    count = composition_summary["用户数"].iloc[i]
    pct = composition_summary["占比"].iloc[i]
    churn_rate = composition_summary["流失率"].iloc[i]
    txt.set_text(f"人数:{count}\n占比:{pct:.1%}\n流失率:{churn_rate:.1%}")
    txt.set_fontsize(7)
    txt.set_color("black")
    txt.set_weight("bold")

# 美化外圈标签
plt.setp(texts, size=12)

ax_composition.set_title("各用户任期阶段构成及流失率", fontsize=14, pad=20)
ax_composition.axis('equal')

# 路径适配：notebooks 与 output 平级
composition_path = Path("../output/day06_visualization/04_composition_chart.png")
composition_path.parent.mkdir(parents=True, exist_ok=True)

fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.close(fig_composition)

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.resolve())

已输出： C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day06_visualization\04_composition_chart.png


### 构成图结论

- 观察：用户整体分布以「0-6 个月」和「13-24 个月」两个阶段为主，两者合计占比超过55%；而流失率呈现明显的 “新用户流失高、老用户流失低” 的特征，其中新用户流失率高达53.5%，是所有阶段中最高的。
- 证据：主要用户分布在：
0-6个月：29.2%
13-24个月：26.1%
7-12个月：28.1%
这三类用户合计占比超过83%，是平台的核心用户群体。
- 边界：该环形图适合展示各用户任期阶段的整体占比分布，能够清晰反映平台用户结构的 “体量特征”；但不适合用来直接比较各阶段的流失率高低，因为流失率和用户数是两个独立指标，放在同一环形中会容易造成误解，更适合用折线图单独呈现流失率趋势。


## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


检查点2通过：4张独立图已生成
检查点3需要结合图表和文字结论人工复核


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# 适配项目结构：notebooks 目录运行，output 在上级目录
ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT.parent / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------- 数据准备：按时间顺序重排 ----------------------
# 1. 定义你需要的正确顺序
ordered_tenure = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# 2. 按正确顺序分组统计
tenure_summary = df.groupby("TenureGroup").agg(
    用户数=("CustomerID", "nunique"),
    流失率=("Churn", "mean")
).reindex(ordered_tenure).reset_index()

# 3. 返现金额数据
return_summary = df[["Tenure", "CashbackAmount", "Churn"]].copy()

# ---------------------- 2×2 子图 ----------------------
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# 全局标题
fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")

# ---------------------- 子图1：任期流失率柱状图（按时间顺序） ----------------------
bars = axes[0,0].bar(
    tenure_summary["TenureGroup"],
    tenure_summary["流失率"],
    color="#1f77b4"
)
# 添加标注：流失率 + 用户数
for idx, row in tenure_summary.iterrows():
    axes[0,0].text(
        idx,
        row["流失率"] + 0.01,
        f"{row['流失率']:.1%}\n(n={row['用户数']})",
        ha="center",
        fontsize=10
    )
axes[0,0].set_title("不同用户任期阶段的流失率对比", fontsize=12)
axes[0,0].set_ylabel("用户流失率")
axes[0,0].grid(axis='y', alpha=0.3)

# ---------------------- 子图2：用户任期-返现散点图 ----------------------
# 流失用户
axes[0,1].scatter(
    return_summary[return_summary["Churn"]==1]["Tenure"],
    return_summary[return_summary["Churn"]==1]["CashbackAmount"],
    color="#d62728",
    alpha=0.7,
    label=f"已流失用户 (n={len(return_summary[return_summary['Churn']==1])})"
)
# 留存用户
axes[0,1].scatter(
    return_summary[return_summary["Churn"]==0]["Tenure"],
    return_summary[return_summary["Churn"]==0]["CashbackAmount"],
    color="#1f77b4",
    alpha=0.7,
    label=f"留存用户 (n={len(return_summary[return_summary['Churn']==0])})"
)
axes[0,1].set_title("用户任期时长与返现金额的关系（按流失状态区分）", fontsize=12)
axes[0,1].set_xlabel("用户任期时长（月）")
axes[0,1].set_ylabel("累计返现金额")
axes[0,1].grid(alpha=0.3)
axes[0,1].legend()

# ---------------------- 子图3：任期流失率折线图（按时间顺序） ----------------------
axes[1,0].plot(
    tenure_summary["TenureGroup"],
    tenure_summary["流失率"],
    marker='o',
    color="#ff7f0e",
    linewidth=2
)
# 添加标注：流失率 + 用户数
for idx, row in tenure_summary.iterrows():
    axes[1,0].text(
        idx,
        row["流失率"] + 0.015,
        f"{row['流失率']:.1%}\n(n={row['用户数']})",
        ha="center",
        fontsize=10
    )
axes[1,0].set_title("用户流失率随任期阶段变化的有序比较", fontsize=12)
axes[1,0].set_xlabel("用户任期阶段（有序）")
axes[1,0].set_ylabel("用户流失率")
axes[1,0].grid(alpha=0.3)

# ---------------------- 子图4：任期构成环形图（按时间顺序） ----------------------
wedges, texts, autotexts = axes[1,1].pie(
    tenure_summary["用户数"],
    labels=tenure_summary["TenureGroup"],
    autopct='',
    startangle=90,
    wedgeprops=dict(width=0.4, edgecolor='white'),
    colors=["#9467bd", "#1f77b4", "#d62728", "#ff7f0e", "#2ca02c"]
)
# 标注：人数 + 占比 + 流失率
for i, txt in enumerate(autotexts):
    row = tenure_summary.iloc[i]
    total = tenure_summary["用户数"].sum()
    pct = row["用户数"] / total
    txt.set_text(f"人数:{row['用户数']}\n占比:{pct:.1%}\n流失率:{row['流失率']:.1%}")
    txt.set_fontsize(8)
    txt.set_color("black")
    txt.set_weight("bold")
axes[1,1].set_title("各用户任期阶段构成及流失率", fontsize=12)
axes[1,1].axis('equal')

# ---------------------- 布局调整与保存 ----------------------
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

# 保存路径：与你作业模板完全一致
summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.close(fig_summary)

# 断言校验
assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.resolve())

已输出： C:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day06_visualization\day06_visualization_summary.png


## 综合发现与局限

1. 综合发现1：用户流失率随任期增长显著下降，新用户是流失重灾区
证据：
从任期分组的柱状图和折线图可见，流失率呈现清晰的递减趋势：
新用户流失率高达53.5%（样本量n=508）
0-6个月用户流失率为25.9%
7-12个月降至9.8%，13-24个月降至6.5%
任职24个月以上的老用户流失率为0%
这说明平台的流失压力主要集中在用户入职初期，随着用户留存时间拉长，其稳定性会显著增强。
2. 综合发现2：投诉是影响用户留存的关键负面因素
证据：
投诉分组的柱状图显示，用户流失率差异巨大：
无投诉用户流失率仅 10.9%
有投诉用户流失率高达 31.7%，是前者的近 3 倍
这表明用户投诉体验与流失行为高度相关，是平台需要重点优化的环节。
3. 综合发现3：返现激励对不同流失状态用户的影响存在差异
证据：
任期时长与返现金额的散点图显示，用户累计返现金额主要集中在 100-300 区间，且已流失用户（红点）多分布在低任期、中返现金额区域，而留存用户（蓝点）分布更广，部分长期用户也获得了较高的返现。
这说明返现金额与用户留存并非简单的正相关，单纯提高返现金额不一定能有效降低流失率。
4. 数据或方法局限：
数据层面：
（1）仅包含 CashbackAmount（返现金额），缺少用户真实消费 GMV、营收数据，无法判断返现对用户消费价值的实际影响。
（2）缺少投诉的具体内容、处理时效等细节数据，无法定位导致用户流失的具体痛点。
分析方法层面：
（1）本次分析为描述性统计，未控制性别、消费频次等混杂变量，因此无法严格证明变量间的因果关系。
（2）数据为截面数据，缺少对用户后续留存行为的长期跟踪，结论仅能反映当前状态，无法预测用户的长期生命周期变化。



## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
# 先导入pandas库
import pandas as pd
from pathlib import Path

# 路径适配
ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT.parent / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TODO：填写5行清单，不得保留“请填写”
chart_manifest = pd.DataFrame([
    {
        "chart_id": "01",
        "file_name": "01_category_bar.png",
        "business_question": "用户投诉状态与流失率的关系对比",
        "chart_type": "bar",
        "key_finding": "有投诉用户流失率（31.7%）显著高于无投诉用户（10.9%），投诉是影响留存的关键负面因素",
        "limitation": "仅展示流失率均值差异，无法反映投诉处理时效、内容对流失的具体影响"
    },
    {
        "chart_id": "02",
        "file_name": "02_behavior_scatter.png",
        "business_question": "用户任期时长与返现金额的关系（按流失状态区分）",
        "chart_type": "scatter",
        "key_finding": "已流失用户多集中在低任期、中返现区间，留存用户分布更广，返现与留存并非简单正相关",
        "limitation": "仅包含累计返现金额，无法判断返现对用户长期消费价值的影响"
    },
    {
        "chart_id": "03",
        "file_name": "03_ordered_line.png",
        "business_question": "用户流失率随任期阶段变化的有序趋势",
        "chart_type": "line",
        "key_finding": "流失率随用户任期增长显著下降，新用户流失率（53.5%）最高，24个月以上用户流失率为0%",
        "limitation": "仅为截面数据的趋势对比，无法跟踪用户后续留存变化，不能验证因果关系"
    },
    {
        "chart_id": "04",
        "file_name": "04_composition_chart.png",
        "business_question": "各用户任期阶段的人数构成及流失率分布",
        "chart_type": "pie_or_bar",
        "key_finding": "用户主要集中在0-6个月和7-12个月阶段，新用户流失率高但占比低，是重点留存优化对象",
        "limitation": "环形图同时展示占比和流失率，易造成信息混淆，无法直观比较不同阶段的流失率差异"
    },
    {
        "chart_id": "05",
        "file_name": "day06_visualization_summary.png",
        "business_question": "电商用户流失影响因素综合分析概览",
        "chart_type": "dashboard",
        "key_finding": "用户流失受任期、投诉行为等多因素影响，新用户和投诉用户是核心流失风险群体",
        "limitation": "仅整合了4张单维图表，未展示变量间的交叉分析，无法反映多因素协同作用的影响"
    },
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)

,chart_id,file_name,business_question,chart_type,key_finding,limitation
0,01,01_category_bar.png,用户投诉状态与流失率的关系对比,bar,有投诉用户流失率（31.7%）显著高于无投诉用户（10.9%），投诉是影响留存的关键负面因素,仅展示流失率均值差异，无法反映投诉处理时效、内容对流失的具体影响
1,02,02_behavior_scatter.png,用户任期时长与返现金额的关系（按流失状态区分）,scatter,已流失用户多集中在低任期、中返现区间，留存用户分布更广，返现与留存并非简单正相关,仅包含累计返现金额，无法判断返现对用户长期消费价值的影响
2,03,03_ordered_line.png,用户流失率随任期阶段变化的有序趋势,line,流失率随用户任期增长显著下降，新用户流失率（53.5%）最高，24个月以上用户流失率为0%,仅为截面数据的趋势对比，无法跟踪用户后续留存变化，不能验证因果关系
3,04,04_composition_chart.png,各用户任期阶段的人数构成及流失率分布,pie_or_bar,用户主要集中在0-6个月和7-12个月阶段，新用户流失率高但占比低，是重点留存优化对象,环形图同时展示占比和流失率，易造成信息混淆，无法直观比较不同阶段的流失率差异
4,05,day06_visualization_summary.png,电商用户流失影响因素综合分析概览,dashboard,用户流失受任期、投诉行为等多因素影响，新用户和投诉用户是核心流失风险群体,仅整合了4张单维图表，未展示变量间的交叉分析，无法反映多因素协同作用的影响


In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")


检查点4通过：第6天成果物完整
下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。
